# Questão 8 - Sistema de recomendação

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

### Caminho base do projeto

In [2]:
BASE_PATH = Path().resolve()

while BASE_PATH.name != "lh-nautical-data-project":
    BASE_PATH = BASE_PATH.parent

print(f"BASE PATH: {BASE_PATH}")

BASE PATH: /media/richard/RichardData/lh-nautical-data-project


Caminhos para as pastas do projeto.

In [3]:
DATA_PATH = BASE_PATH / "data"

RAW_PATH = DATA_PATH / "raw"
STAGING_PATH = DATA_PATH / "staging"
INTERMEDIATE_PATH = DATA_PATH / "intermediate"
MARTS_PATH = DATA_PATH / "marts"

SQL_PATH = BASE_PATH / "sql"
IMAGES_PATH = BASE_PATH / "imagens"

### Carregar dataset

In [4]:


df_dataset = pd.read_csv(RAW_PATH/"vendas_2023_2024.csv")

df_dataset.head()


,id,id_client,id_product,qtd,total,sale_date
0,0,42,105,11,3405.0,2023-09-10
1,1,3,136,9,16873.9,15-09-2024
2,2,25,139,7,9475.3,2024-08-13
3,4,20,23,5,55893.0,2023-02-03
4,5,8,57,4,451403.9,2024-02-12


Criar matriz binária (usuário x produto)

In [5]:
matriz = (
    df_dataset.groupby(['id_client', 'id_product'])
    .size()
    .unstack(fill_value=0)
)

Converter para binário (1 = comprou, 0 = não comprou)

In [6]:
matriz = (matriz > 0).astype(int)

matriz.head()

id_product,1,2,3,4,5,6,7,8,9,10,...,141,142,143,144,145,146,147,148,149,150
id_client,,,,,,,,,,,,,,,,,,,,,
1,1,0,1,1,1,1,0,0,0,0,...,1,1,0,1,1,1,1,1,0,0
2,0,1,1,0,1,1,1,1,1,1,...,1,1,1,1,1,1,0,1,1,1
3,1,1,1,1,1,1,1,0,1,1,...,0,1,1,1,0,1,1,1,1,1
4,1,1,0,1,1,0,1,0,0,1,...,1,1,1,1,0,0,1,0,1,1
5,1,1,0,1,1,1,1,1,1,1,...,1,1,1,1,0,1,1,1,1,0


Calculando produto x produto

Transpondo, agora linhas = produtos, colunas = clientes

In [7]:
matriz_produto = matriz.T

Calcular similaridade de cosseno

In [8]:

similaridade = cosine_similarity(matriz_produto)


Transformar em DataFrame

In [9]:
df_similaridade = pd.DataFrame(
    similaridade,
    index=matriz_produto.index,
    columns=matriz_produto.index
)

df_similaridade.head()

id_product,1,2,3,4,5,6,7,8,9,10,...,141,142,143,144,145,146,147,148,149,150
id_product,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.775058,0.737865,0.810191,0.748331,0.769484,0.775058,0.727825,0.711512,0.698771,...,0.795133,0.753819,0.775058,0.769484,0.721605,0.775058,0.872082,0.721605,0.727825,0.775058
2,0.775058,1.000000,0.704295,0.757865,0.714286,0.712931,0.771429,0.750290,0.788811,0.687256,...,0.795192,0.778078,0.771429,0.767772,0.685714,0.742857,0.712931,0.628571,0.750290,0.742857
3,0.737865,0.704295,1.000000,0.800641,0.704295,0.865181,0.732467,0.712396,0.777778,0.707107,...,0.675923,0.739795,0.732467,0.757033,0.704295,0.788811,0.729996,0.732467,0.739795,0.704295
4,0.810191,0.757865,0.800641,1.000000,0.757865,0.753310,0.757865,0.789747,0.773953,0.735980,...,0.831239,0.789747,0.730798,0.779287,0.730798,0.730798,0.805263,0.757865,0.763422,0.757865
5,0.748331,0.714286,0.704295,0.757865,1.000000,0.795192,0.742857,0.694713,0.760639,0.627495,...,0.795192,0.722501,0.714286,0.685511,0.685714,0.714286,0.767772,0.657143,0.722501,0.714286


Agora para descobrir o id do "GPS Garmin Vortex Maré Drift"

In [10]:
df_produtos = pd.read_csv(RAW_PATH/"produtos_raw.csv")

df_produtos.head()

,name,price,code,actual_category
0,Transponder AIS Maré Magnum,R$ 33122.52,1,ELETRONICOS
1,Transponder Furuno Marlin,R$ 13998.15,2,ELETRONICOS
2,Radar Furuno Pulse Leviathan,R$ 9024.19,3,E L E T R Ô N I C O S
3,Rádio AIS Hydro Tidal Zen,R$ 3381.88,4,Eletrunicos
4,Piloto Automático Furuno Storm,R$ 23669.01,5,Eletronicoz


In [11]:
df_produtos[df_produtos['name'].str.contains("GPS Garmin", case=False, na=False)]

,name,price,code,actual_category
26,GPS Garmin Vortex Maré Drift,R$ 16511.45,27,ELETRONICOS


GPS Garmin Vortex Maré Drift tem o id 27

In [12]:
produto_id = 27

Pegar similaridades do produto

In [13]:
similares = df_similaridade.loc[produto_id]

Ordenar do maior para o menor

In [14]:
ranking = similares.sort_values(ascending=False)

Remover o próprio produto "GPS Garmin Vortex Maré Drift" (similaridade = 1)

In [15]:
ranking = ranking.drop(produto_id)

Top 5 produtos mais similares

In [16]:
top5 = ranking.head(5)

top5

id_product
94     0.869626
11     0.868037
35     0.853913
115    0.850000
1      0.850000
Name: 27, dtype: float64

Top 5 produtos mais similares com seus nomes

In [17]:
top5_df = top5.reset_index()
top5_df.columns = ['id_produto', 'similaridade']

top5_com_nome = top5_df.merge(
    df_produtos[['code', 'name']],
    left_on='id_produto',
    right_on='code',
    how='left'
)
top5_com_nome = top5_com_nome[['id_produto', 'name', 'similaridade']]

top5_com_nome

,id_produto,name,similaridade
0,94,Motor de Popa Volvo Magnum 276HP,0.869626
1,11,GPS Furuno Swift Leviathan Poseidon,0.868037
2,35,Radar Furuno Swift,0.853913
3,115,Cabo de Nylon Delta Force Magnum Leviathan,0.850000
4,1,Transponder AIS Maré Magnum,0.850000


Questão 8.2 - Validação

Qual é o id _produto com MAIOR similaridade ao 
"GPS Garmin Vortex Maré Drift"?

Resposta:
94

In [18]:
top5.index[0]

np.int64(94)

O produto com a maior similaridade é o "Motor de Popa Volvo Magnum 276HP" que tem o id 94	

com 0.869626 de similaridade.

Questão 8.3 - Explique:
1. Como a matriz foi construída?
2 . O que significa a similaridade de cosseno nesse
contexto?
3. Uma limitação desse método de recomendação.

Construi a matriz foi usando usuário × produto, onde cada linha representa um cliente e cada coluna um produto. Os valores são 1 se o cliente comprou o produto ao menos uma vez, 0 caso contrário, sem levar em conta a quantidade.

A similaridade mede o quanto dois produtos são comprados pelos mesmos clientes. Quanto mais próximo de 1, maior a semelhança no comportamento de compra.

Uma limitação é que o método considera apenas se houve ou não uma compra, se houvessem mais variaveis como quantidade de compra, contexto, características dos produtos e dos clientes, como demografia ou mesmo compras em sites de concorrentes poderia ser usado algo mais robusto como o KNN por exemplo.